In [23]:
import json
import pandas as pd

from pathlib import Path
from collections import Counter

# Project directory
BASE_DIR = Path.cwd()

# Dataset files
DATASET_FILE = BASE_DIR / "malay_cleaned_dataset" / "malay_evaluation_dataset.jsonl"
EXCLUDED_FILE = BASE_DIR / "malay_cleaned_dataset" / "malay_excluded_records.jsonl"
REPORT_FILE = BASE_DIR / "malay_cleaned_dataset" / "malay_dataset_validation_report.txt"

print("Dataset exists:", DATASET_FILE.exists())
print("Excluded file exists:", EXCLUDED_FILE.exists())
print("Report exists:", REPORT_FILE.exists())

Dataset exists: True
Excluded file exists: True
Report exists: True


In [24]:
records = []
json_errors = []

with open(DATASET_FILE, "r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as e:
            json_errors.append({
                "line": line_number,
                "error": str(e)
            })

print("Total records loaded:", len(records))
print("JSON parsing errors:", len(json_errors))

Total records loaded: 3448
JSON parsing errors: 0


In [25]:
required_keys = {
    "domain",
    "input",
    "reference"
}

missing_keys = []

for index, record in enumerate(records, start=1):

    missing = required_keys - set(record.keys())

    if missing:
        missing_keys.append({
            "record": index,
            "missing_keys": list(missing)
        })

print("Records with missing keys:", len(missing_keys))

Records with missing keys: 0


In [26]:
EXPECTED_SCHEMA = {
    "Campuran Kod": {
        "input": ["korpus ujaran online"],
        "reference": [
            "ujaran tidak formal",
            "ujaran formal"
        ]
    },

    "Penukaran Kod": {
        "input": ["soalan"],
        "reference": [
            "jawapan formal",
            "jawapan tidak formal"
        ]
    },

    "Terjemahan": {
        "input": ["soalan (bahasa sumber)"],
        "reference": ["jawapan (bahasa sasaran)"]
    },

    "Sentimen News": {
        "input": ["teks"],
        "reference": ["sifat sentimen"]
    },

    "Sentimen": {
        "input": ["soalan"],
        "reference": ["jawapan"]
    }
}

schema_errors = []

for index, record in enumerate(records, start=1):

    domain = record.get("domain")

    if domain not in EXPECTED_SCHEMA:
        schema_errors.append({
            "record": index,
            "error": "Unknown domain"
        })
        continue

    expected = EXPECTED_SCHEMA[domain]

    actual_input = list(record.get("input", {}).keys())
    actual_reference = list(record.get("reference", {}).keys())

    if set(actual_input) != set(expected["input"]):
        schema_errors.append({
            "record": index,
            "domain": domain,
            "error": "Input schema mismatch",
            "actual": actual_input
        })

    if set(actual_reference) != set(expected["reference"]):
        schema_errors.append({
            "record": index,
            "domain": domain,
            "error": "Reference schema mismatch",
            "actual": actual_reference
        })

print("Schema errors:", len(schema_errors))

Schema errors: 0


In [27]:
empty_value_records = []

for index, record in enumerate(records, start=1):

    domain = record["domain"]

    input_values = list(record["input"].values())
    reference_values = list(record["reference"].values())

    # Input wajib sentiasa lengkap
    invalid_input = any(
        value is None or not isinstance(value, str) or not value.strip()
        for value in input_values
    )

    if invalid_input:
        empty_value_records.append(index)
        continue

    # Campuran Kod: sekurang-kurangnya satu reference mesti tersedia
    if domain == "Campuran Kod":

        available_references = [
            value for value in reference_values
            if isinstance(value, str) and value.strip()
        ]

        if not available_references:
            empty_value_records.append(index)

    else:
        # Domain lain: semua reference wajib lengkap
        invalid_reference = any(
            value is None or not isinstance(value, str) or not value.strip()
            for value in reference_values
        )

        if invalid_reference:
            empty_value_records.append(index)

print("Records with empty/invalid required values:",
      len(empty_value_records))

Records with empty/invalid required values: 0


In [28]:
seen = set()
duplicate_records = []

for index, record in enumerate(records, start=1):

    signature = json.dumps(
        {
            "domain": record["domain"],
            "input": record["input"],
            "reference": record["reference"]
        },
        ensure_ascii=False,
        sort_keys=True
    )

    if signature in seen:
        duplicate_records.append(index)

    seen.add(signature)

print("Remaining exact duplicates:", len(duplicate_records))

Remaining exact duplicates: 0


In [29]:
domain_counts = Counter(
    record["domain"] for record in records
)

validation_results = {
    "JSON parsing": len(json_errors) == 0,
    "Required keys": len(missing_keys) == 0,
    "Domain schema": len(schema_errors) == 0,
    "Empty values": len(empty_value_records) == 0,
    "Exact duplicates": len(duplicate_records) == 0,
    "Record count": len(records) == 3448
}

print("=" * 55)
print("FINAL DATASET VALIDATION REPORT")
print("=" * 55)

for check, passed in validation_results.items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

print("\nDOMAIN DISTRIBUTION")
display(
    pd.DataFrame(
        domain_counts.items(),
        columns=["Domain", "Records"]
    )
)

print("\nTotal records:", len(records))

print("\nOVERALL STATUS:",
      "PASS" if all(validation_results.values()) else "FAIL")

FINAL DATASET VALIDATION REPORT
JSON parsing: PASS
Required keys: PASS
Domain schema: PASS
Empty values: PASS
Exact duplicates: PASS
Record count: PASS

DOMAIN DISTRIBUTION


,Domain,Records
0,Campuran Kod,275
1,Penukaran Kod,799
2,Terjemahan,712
3,Sentimen News,980
4,Sentimen,682



Total records: 3448

OVERALL STATUS: PASS
